# Reuters C50 - Klasifikacija autora novinskih tekstova
## 03 - Vizualizacija podataka u 2D i 3D prostoru

Pošto tekstualni podaci nemaju direktno grafičku reprezentaciju, koristimo tehnike redukcije dimenzionalnosti
(**PCA** i **t-SNE**) da projektujemo visoko-dimenzionalne TF-IDF vektore u 2D i 3D prostor.

### Ćelija 1 - Uvoz i učitavanje preprocesiranih podataka

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from mpl_toolkits.mplot3d import Axes3D

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import LabelEncoder

SLIKE_PATH   = '/home/zagorka/Desktop/ip2Projekat/slike'
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('/home/zagorka/Desktop/ip2Projekat/reuters_preprocessed.csv')

train_df = df[df['split'] == 'C50train'].reset_index(drop=True)
test_df  = df[df['split'] == 'C50test'].reset_index(drop=True)

print(f'Train: {len(train_df)} | Test: {len(test_df)}')
print(f'Autori: {df["author"].nunique()}')

> **Zaključak:** Preprocesirani podaci su uspešno učitani (5000 redova). Dataset je ravnomerno podeljen — 2500 train i 2500 test dokumenata, sa 50 klasa (autora). Koristimo ceo dataset (train + test) za vizualizaciju kako bismo videli prostornu raspodelu svih dokumenata.

### Ćelija 2 - TF-IDF vektorizacija (za vizualizaciju)

In [ ]:
# TF-IDF na celom datasetu (za vizualizaciju koristimo sve dokumente)
tfidf_viz = TfidfVectorizer(max_features=5000, ngram_range=(1,1), sublinear_tf=True)
X_all     = tfidf_viz.fit_transform(df['text_clean'])
y_all     = df['author'].values

# Label encoder za boje
le     = LabelEncoder()
y_enc  = le.fit_transform(y_all)
colors = cm.tab20(np.linspace(0, 1, 50))

print(f'TF-IDF matrica: {X_all.shape}')
print(f'Broj autora (klasa): {len(le.classes_)}')
print(f'Sparsnost matrice: {(1 - X_all.nnz / (X_all.shape[0]*X_all.shape[1]))*100:.1f}%')

> **Zaključak:** TF-IDF matrica ima dimenziju **5000 × 5000** — 5000 dokumenata i 5000 najfrekventnijih unigramskih atributa. Matrica je izuzetno retka (**96.8% nula**), što je tipično za tekstualne podatke — svaki dokument koristi samo mali deo ukupnog rečnika. Ovakvu retku matricu nije moguće direktno podati u standardni PCA (koji zahteva gustu matricu), pa koristimo TruncatedSVD kao predkorak koji radi direktno na retkim matricama.

### Ćelija 3 - SVD redukcija dimenzionalnosti (predkorak za PCA/t-SNE)

In [ ]:
# TruncatedSVD radi na sparse matricama — redukujemo na 100 dimenzija
print('SVD redukcija: 5000 → 100 dimenzija...')
svd = TruncatedSVD(n_components=100, random_state=RANDOM_STATE)
X_svd = svd.fit_transform(X_all)

explained = svd.explained_variance_ratio_.sum() * 100
print(f'Objašnjena varijansa (100 komponenti): {explained:.1f}%')
print(f'Dimenzija posle SVD: {X_svd.shape}')

# Scree plot - koliko varijanse hvata svaka komponenta
fig, ax = plt.subplots(figsize=(10, 4))
cumvar = np.cumsum(svd.explained_variance_ratio_) * 100
ax.plot(range(1, 101), cumvar, 'o-', color='steelblue', markersize=3, linewidth=1.5)
ax.axhline(80, color='red',   linestyle='--', label='80% varijanse')
ax.axhline(90, color='green', linestyle='--', label='90% varijanse')
ax.set_title('Kumulativna objašnjena varijansa (TruncatedSVD)', fontweight='bold')
ax.set_xlabel('Broj SVD komponenti')
ax.set_ylabel('Kumulativna varijansa (%)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SLIKE_PATH, '07_svd_scree.png'))
plt.show()
print('Slika sacuvana: 07_svd_scree.png')

> **Zaključak:** Sa 100 SVD komponenti objašnjavamo samo **27.2%** ukupne varijanse originalnih 5000 atributa. Ovo je karakteristično za visoko-dimenzionalne tekstualne podatke — informacija je rasuta po velikom broju dimenzija i teško se kompresuje. Scree plot pokazuje da kriva raste postepeno bez strme "laktične" tačke, što znači da ni jedna mala grupa dimenzija ne dominira informacijom. Uprkos tome, 100 dimenzija je dovoljan predkorak za PCA i t-SNE vizualizaciju.

### Ćelija 4 - PCA 2D vizualizacija

In [ ]:
pca2 = PCA(n_components=2, random_state=RANDOM_STATE)
X_2d = pca2.fit_transform(X_svd)

fig, ax = plt.subplots(figsize=(13, 10))
for i, author in enumerate(le.classes_):
    mask = y_enc == i
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               s=18, alpha=0.75, color=colors[i], label=author)

ax.set_title('2D PCA vizualizacija — Reuters C50\n(TF-IDF → SVD(100) → PCA(2))',
             fontsize=13, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}% var.)')
ax.set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}% var.)')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=6, ncol=2, framealpha=0.8)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(os.path.join(SLIKE_PATH, '08_pca_2d.png'), bbox_inches='tight')
plt.show()
print(f'PC1 varijansa: {pca2.explained_variance_ratio_[0]*100:.2f}%')
print(f'PC2 varijansa: {pca2.explained_variance_ratio_[1]*100:.2f}%')
print('Slika sacuvana: 08_pca_2d.png')

> **Zaključak:** PC1 objašnjava **6.30%**, a PC2 **3.53%** varijanse — zajedno svega 9.83%. PCA 2D projekcija prikazuje značajno preklapanje klasa bez jasnih granica između autora, što je očekivano: linearna PCA nije dovoljno moćna da razdvoji 50 autorskih stilova na osnovu samo dve dimenzije. Ipak, određeni autori mogu biti vidljivi kao odvojeni oblaci na rubovima prostora — ti autori imaju najdistinktivniji vokabular koji se razlikuje od ostatka korpusa.

### Ćelija 5 - PCA 3D vizualizacija

In [ ]:
pca3 = PCA(n_components=3, random_state=RANDOM_STATE)
X_3d = pca3.fit_transform(X_svd)

fig = plt.figure(figsize=(13, 9))
ax3 = fig.add_subplot(111, projection='3d')

for i, author in enumerate(le.classes_):
    mask = y_enc == i
    ax3.scatter(X_3d[mask, 0], X_3d[mask, 1], X_3d[mask, 2],
                s=12, alpha=0.7, color=colors[i], label=author)

ax3.set_title('3D PCA vizualizacija — Reuters C50', fontsize=12, fontweight='bold')
ax3.set_xlabel(f'PC1 ({pca3.explained_variance_ratio_[0]*100:.1f}%)')
ax3.set_ylabel(f'PC2 ({pca3.explained_variance_ratio_[1]*100:.1f}%)')
ax3.set_zlabel(f'PC3 ({pca3.explained_variance_ratio_[2]*100:.1f}%)')
ax3.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=5, ncol=2)
plt.tight_layout()
plt.savefig(os.path.join(SLIKE_PATH, '09_pca_3d.png'), bbox_inches='tight')
plt.show()
total3 = sum(pca3.explained_variance_ratio_)*100
print(f'Ukupna varijansa PC1+PC2+PC3: {total3:.2f}%')
print('Slika sacuvana: 09_pca_3d.png')

> **Zaključak:** Tri PCA komponente zajedno objašnjavaju samo **12.94%** varijanse. Iako treća dimenzija dodaje nešto više informacije u odnosu na 2D (9.83% → 12.94%), 3D projekcija i dalje prikazuje značajno preklapanje klasa. Ipak, u 3D prostoru je vidljivija određena grupisana struktura — neke grupe autora formiraju labave klastere. Ovo potvrđuje da linearno razdvajanje 50 autorskih stilova zahteva znatno više od 2–3 dimenzije.

### Ćelija 6 - t-SNE 2D vizualizacija

In [ ]:
print('t-SNE u toku (može potrajati 2-5 minuta)...')
tsne = TSNE(n_components=2, perplexity=30, max_iter=1000,
            random_state=RANDOM_STATE, verbose=1)
X_tsne = tsne.fit_transform(X_svd)

fig, ax = plt.subplots(figsize=(13, 10))
for i, author in enumerate(le.classes_):
    mask = y_enc == i
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
               s=20, alpha=0.85, color=colors[i], label=author)

ax.set_title('t-SNE 2D vizualizacija — Reuters C50\n(perplexity=30, max_iter=1000)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('t-SNE dimenzija 1')
ax.set_ylabel('t-SNE dimenzija 2')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=6, ncol=2)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(os.path.join(SLIKE_PATH, '10_tsne_2d.png'), bbox_inches='tight')
plt.show()
print('Slika sacuvana: 10_tsne_2d.png')

> **Zaključak:** t-SNE vizualizacija pokazuje znatno bolju separaciju klasa od PCA — vidljivi su odvojeni klasteri za deo autora, što znači da njihovi tekstualni stilovi imaju distinktivne lokalne strukture u prostoru atributa. Za razliku od PCA koji je linearan, t-SNE je nelinearna metoda koja čuva lokalne susedstvene odnose i bolje otkriva grupisanje unutar visoko-dimenzionalnih podataka. Preklapanje i dalje postoji (50 autora je mnogo za 2D prostor), ali je grupna struktura daleko uočljivija nego kod PCA.

### Ćelija 7 - Poređenje PCA vs t-SNE separabilnosti klasa

In [ ]:
# Prikazujemo oba metoda jedan pored drugog za 10 nasumičnih autora
np.random.seed(RANDOM_STATE)
sample_authors = np.random.choice(le.classes_, size=10, replace=False)
sample_colors  = cm.tab10(np.linspace(0, 1, 10))

mask_sample = np.isin(y_all, sample_authors)
y_sample    = y_all[mask_sample]
X_2d_s     = X_2d[mask_sample]
X_tsne_s   = X_tsne[mask_sample]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ci, author in enumerate(sample_authors):
    m = y_sample == author
    axes[0].scatter(X_2d_s[m,0],   X_2d_s[m,1],   s=25, alpha=0.8, color=sample_colors[ci], label=author)
    axes[1].scatter(X_tsne_s[m,0], X_tsne_s[m,1], s=25, alpha=0.8, color=sample_colors[ci], label=author)

axes[0].set_title('PCA 2D (10 autora)', fontweight='bold')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
axes[0].legend(fontsize=8, loc='best'); axes[0].grid(alpha=0.2)

axes[1].set_title('t-SNE 2D (10 autora)', fontweight='bold')
axes[1].set_xlabel('Dim 1'); axes[1].set_ylabel('Dim 2')
axes[1].legend(fontsize=8, loc='best'); axes[1].grid(alpha=0.2)

plt.suptitle('PCA vs t-SNE — separabilnost klasa (10 autora)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(SLIKE_PATH, '11_pca_vs_tsne.png'), bbox_inches='tight')
plt.show()
print('Slika sacuvana: 11_pca_vs_tsne.png')
print(f'Prikazani autori: {list(sample_authors)}')

> **Zaključak:** Poređenje PCA i t-SNE na 10 nasumičnih autora (JanLopatka, RobinSidel, MarkBendeich, TanEeLyn, JoeOrtiz, ToddNissen, LydiaZajc, KouroshKarimkhany, MatthewBunce, JonathanBirt) jasno ilustruje prednost t-SNE: dok PCA prikazuje autore kao isprepletane oblake tačaka bez jasnih granica, t-SNE ih razdvaja u kompaktnije, prostorno odvojene grupe. Ovo vizuelno potvrđuje da su autorski stilovi dovoljno distinktivni da ih algoritmi klasifikacije mogu razlikovati — posebno oni koji rade u višedimenzionalnom prostoru (SVM, Logistička regresija), a ne samo u 2D projekciji.